In [6]:
cfg <- list(minimo = 200, grupo = "Pessoa")

In [8]:
docs   <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/docs.rds"))))
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))
ix     <- montar(docs, cfg)
estado()

Warning message in readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds")))):
“cannot open URL 'https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/estrutura/banco-de-dados/origem.rds': HTTP status was '404 Not Found'”


ERROR: Error in readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds")))): cannot open the connection to 'https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/estrutura/banco-de-dados/origem.rds'


In [10]:
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"

# O config.R está na raiz (conforme o print):
download.file(paste0(REPO, "config.R"), "config.R")
source("config.R")

# Lendo ambos os arquivos rds de dentro da pasta estrutura/banco-de-dados/:
docs   <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/docs.rds"))))
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))

ix     <- montar(docs, cfg)
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 02:59 · R version 4.6.1 (2026-06-24) · motor02 v1 2026-10-01 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$grupo` | character | 1 | <nome do grupo> |
| `docs` | character, nomeado | 64 | d1 São Vicente, oficialmente E..., d2 São Vicente marca o início ..., d3 Com um PIB de mais de 6 bil..., d4 São Vicente preserva divers..., d5 A Ilha de Gohayó foi descob..., d6 Nas primeiras décadas do sé..., ... |
| `ix$tokens` | list | 64 | elementos: d1, d2, d3, d4, d5, d6, d7, d8, ... |
| `ix$vocab` | character | 2136 | -, –, —, "berço, "bica, "biquinha"., ... |
| `ix$tf` | matrix | 2136 × 64 | linhas: -, –, —, ... \| colunas: d1, d2, d3, d4, ... |
| `ix$N` | integer | 1 | 64 |
| `ix$df` | numeric, nomeado | 2136 | - 2, – 2, — 2, "berço 1, "bica 1, "biquinha". 1, ... |
| `ix$idf_tfidf` | numeric, nomeado 

In [11]:
cosseno <- function(a, b) {
  den <- sqrt(sum(a^2)) * sqrt(sum(b^2))
  if (den == 0) return(0)
  sum(a * b) / den
}

norm_cols <- function(m) {
  nrm <- sqrt(colSums(m^2))
  nrm[nrm == 0] <- 1
  sweep(m, 2, nrm, "/")
}

In [12]:
dim(ix$w)                                            # termos x documentos
normas <- sqrt(colSums(ix$w^2))                      # a norma de cada documento
round(sort(normas, decreasing = TRUE)[1:3], 2)       # os três "mais longos"
maior <- names(sort(normas, decreasing = TRUE))[1]   # o nome do de maior norma
origem[[maior]]                                      # de que artigo ele veio

[1] 2136   64

d36   d46   d31 
61.54 54.29 46.31

[1] "Martim Afonso de Sousa"

In [13]:
length(ix$tokens[[maior]])                   # quantas palavras ele tem
round(mean(sapply(ix$tokens, length)))       # quantas tem um documento médio do corpus
sum(ix$df[ix$tokens[[maior]]] == 1)          # quantas das palavras dele só aparecem nele
wn <- norm_cols(ix$w)                        # as colunas com comprimento 1
table(round(colSums(wn^2), 2))               # todos 1?

[1] 277

[1] 86

[1] 95


 1 
64 

In [14]:
vetor_consulta <- function(termos, vocab, idf) {            # Modulo 6 da teoria
  q <- as.integer(table(factor(termos, levels = vocab)))    # contagens; termo fora do vocabulario sumira
  q * idf                                                   # a regua do corpus
}                                                           # fim da funcao

ranking_cosseno <- function(consulta, ix) {                           # Modulo 7 da teoria
  qw <- vetor_consulta(tokenizar(consulta), ix$vocab, ix$idf_tfidf)   # consulta no espaco do corpus
  s  <- apply(ix$w, 2, function(d) cosseno(qw, d))                    # um cosseno por documento
  sort(s, decreasing = TRUE)                                          # do mais parecido ao menos
}                                                                     # fim da funcao

In [15]:
consulta <- "monumentos historicos centro"    # Testem uma de cada vez (troquem a string)
tokenizar(consulta) %in% ix$vocab            # Deve dar TRUE para todos os termos
r <- ranking_cosseno(consulta, ix)           # Ranking completo
round(r[1:5], 3)                             # 5 primeiros colocados
melhor <- names(r)[1]                        # Nome do 1º lugar
origem[[melhor]]                             # De que artigo veio o 1º lugar
docs[[melhor]]                               # Texto do 1º lugar

[1] FALSE FALSE  TRUE

d6    d1    d2    d3    d4 
0.142 0.000 0.000 0.000 0.000

[1] "São Vicente (São Paulo)"

[1] "Nas primeiras décadas do século XVI, existiu, onde hoje é o centro de São Vicente, um povoado, fundado pelo enigmático Bacharel de Cananeia, um degredado português que se fixou no litoral de São Paulo provavelmente entre 1498 e 1502. Em 1526, essa povoação descrita como sendo \"de dez ou doze casas, uma feita de pedra com seus telhados e uma torre para defesa contra os índios no tempo de necessidade. Estão providos de coisas da terra, de galinhas de Espanha e de porcos, com muita abundância de hortaliças\". Dessa forma, o Bacharel de Cananeia pode ser considerado o verdadeiro fundador de São Vicente. "

In [17]:
consulta <- "fundacao vila martim afonso"    # Testem uma de cada vez (troquem a string)
tokenizar(consulta) %in% ix$vocab            # Deve dar TRUE para todos os termos
r <- ranking_cosseno(consulta, ix)           # Ranking completo
round(r[1:5], 3)                             # 5 primeiros colocados
melhor <- names(r)[1]                        # Nome do 1º lugar
origem[[melhor]]                             # De que artigo veio o 1º lugar
docs[[melhor]]                               # Texto do 1º lugar

[1] FALSE  TRUE  TRUE  TRUE

d27    d7   d28   d38   d13 
0.163 0.124 0.122 0.101 0.099

[1] "Martim Afonso de Sousa"

[1] "Martim Afonso de Sousa descendia dos Sousa Chichorro, cujo varão da geração inicial foi Martim Afonso Chichorro, filho bastardo de D. Afonso III, o qual foi elevado a altas posições sociais, políticas e económicas, no reinado de seu meio-irmão D. Dinis. O mesmo Martim Afonso Chichorro teve um filho homónimo com Inês Lourenço de Valadares. Este último teve um filho chamado Vasco Martins de Sousa Chichorro que já em avançada idade (naquele tempo) participou nas Cortes de Coimbra, e contribuiu para aclamação de D. João I. Seu filho, chamado, Martim Afonso de Sousa, combateu na Batalha de Aljubarrota na qual integrou a Ala dos Namorados, composta por cavaleiros que juraram que, se escapassem com vida da batalha, correriam imediatamente para os braços das suas namoradas."

In [18]:
consulta <- "ponte pensil arquitetura"    # Testem uma de cada vez (troquem a string)
tokenizar(consulta) %in% ix$vocab            # Deve dar TRUE para todos os termos
r <- ranking_cosseno(consulta, ix)           # Ranking completo
round(r[1:5], 3)                             # 5 primeiros colocados
melhor <- names(r)[1]                        # Nome do 1º lugar
origem[[melhor]]                             # De que artigo veio o 1º lugar
docs[[melhor]]                               # Texto do 1º lugar

[1]  TRUE FALSE FALSE

d57   d53   d64   d62   d59 
0.244 0.208 0.200 0.170 0.165

[1] "Ponte Pênsil de São Vicente"

[1] "Os pontões da ponte foram instalados em 1911. O material para a ponte foi importado da Alemanha e chegou em dez navios. Foi inaugurada em 21 ou 24 de maio de 1914, com Washington Luís (então prefeito de São Paulo) sendo a primeira pessoa a atravessar a ponte em um carro. Estava acompanhado de sua comitiva."

In [21]:
consulta <- "praias"                         # Coloquem a consulta de teste escolhida
tokenizar(consulta) %in% ix$vocab            # O termo existe no vocabulário? (FALSE = vai sumir)
round(ranking_cosseno(consulta, ix)[1:5], 3) # O ranking obtido
grep("^prai", ix$vocab, value = TRUE)        # Ver formas parecidas existentes no vocabulário
round(ix$idf_tfidf[c("de", "do", "da")], 2)  # Verificar o idf de palavras vazias (se testarem esse caminho)

[1] FALSE

d1 d2 d3 d4 d5 
 0  0  0  0  0

[1] "praia"    "praia."   "praias,"  "prainha."

de   do   da 
0.02 0.31 0.23

In [22]:
list.files()

[1] "00_FICHA_PROJETO.md"          "aula02_parteD_consolidado.md"
[3] "config.R"                     "sample_data"

In [23]:
anexar_estado("00_FICHA_PROJETO.md")             # O estado atualizado do R vai para a ficha
anexar_estado("aula02_parteD_consolidado.md")    # E para o consolidado do grupo

Estado do R anexado a 00_FICHA_PROJETO.md (19 linhas de objetos).

Estado do R anexado a aula02_parteD_consolidado.md (19 linhas de objetos).

